## Process Lab Result Data

In [ ]:
import os

import pandas as pd
import numpy as np
import re

from math import sqrt

from datetime import date

data_path = os.path.join("..", "data")
raw_data_path = os.path.join(data_path, 'raw_data/')
processed_data_path = os.path.join(data_path, 'processed_data/')

## Load Data

### Examinations Data

In [ ]:
examinations_df = pd.read_csv(os.path.join(raw_data_path, "20251204_examinations_search_results.csv"))

examinations_df['measurement_date'] = pd.to_datetime(examinations_df['measurement_date'])
examinations_df['measurement_datetime'] = pd.to_datetime(examinations_df['measurement_datetime'])

### Date Breakdown Data

In [ ]:
dates_breakdown_df = pd.read_csv(os.path.join(data_path, "ckd_patients_datebreakdown.csv"))

dates_breakdown_df['inclusion_date'] = pd.to_datetime(dates_breakdown_df['inclusion_date']).dt.date
dates_breakdown_df['endpoint_date'] = pd.to_datetime(dates_breakdown_df['endpoint_date']).dt.date
dates_breakdown_df['start_date'] = pd.to_datetime(dates_breakdown_df['start_date']).dt.date
dates_breakdown_df['end_date'] = pd.to_datetime(dates_breakdown_df['end_date']).dt.date

## Refine Examination Results

In [ ]:
def value_conversion(row):
    if row['measure'] == 'Height' and row['unit'] != 'cm':
        return float(row['value'])*2.54
    else:
        return row['value']

In [ ]:
cols = ['master_person_id', 'measurement_date', 'measurement_datetime', 'measure', 'value', 'unit']

examinations_refined_df = dates_breakdown_df.merge(examinations_df[cols], how='right', on='master_person_id')

start_date_filter = (examinations_refined_df['measurement_date']>=examinations_refined_df['start_date'])
end_date_filter = (examinations_refined_df['measurement_date']<=examinations_refined_df['end_date'])

examinations_refined_df = examinations_refined_df[start_date_filter&end_date_filter].drop_duplicates().reset_index(drop=True)

examinations_refined_df['grouping'] = examinations_refined_df['grouping'].astype(int)
examinations_refined_df['measure'] = examinations_refined_df['measure'].str.replace(' ', '_')

examinations_refined_df['value'] = examinations_refined_df.apply(lambda x: value_conversion(x), axis=1)

del start_date_filter, end_date_filter, cols, value_conversion

In [ ]:
examinations_pivoted_df = examinations_refined_df.pivot_table(values='value',
                                                              index=['master_person_id', 'grouping', 'measurement_datetime'],
                                                              columns=['measure'],
                                                              aggfunc='first').reset_index()

del examinations_refined_df

### Structure & Properly Populate Blood Pressure Results

In [ ]:
def bp_diasyloic(row):
    if pd.isnull(row['BP_Diastolic']) and not pd.isnull(row['Blood_Pressure']):
        return row['Blood_Pressure_Breakdown'][1]
    else:
        return row['BP_Diastolic']

In [ ]:
def bp_systolic(row):
    if pd.isnull(row['BP_Systolic']) and not pd.isnull(row['Blood_Pressure']):
        return row['Blood_Pressure_Breakdown'][0]
    else:
        return row['BP_Systolic']

In [ ]:
def arterial_pressure(row):
    if not pd.isnull(row['BP_Systolic']) and not pd.isnull(row['BP_Diastolic']):
        return row['BP_Diastolic'] + (1/3)*(row['BP_Systolic']-row['BP_Diastolic'])
    else:
        return np.NaN

In [ ]:
examinations_pivoted_df['Blood_Pressure'] = examinations_pivoted_df['Blood_Pressure'].astype(str)
examinations_pivoted_df['Blood_Pressure_Breakdown'] = examinations_pivoted_df['Blood_Pressure'].apply(lambda x: np.NaN if x=='nan' else x.split('/'))
examinations_pivoted_df['Blood_Pressure'] = examinations_pivoted_df['Blood_Pressure'].apply(lambda x: np.NaN if x=='nan' else x)

examinations_pivoted_df['BP_Diastolic'] = examinations_pivoted_df.apply(lambda row: bp_diasyloic(row), axis=1)
examinations_pivoted_df['BP_Diastolic'] = pd.to_numeric(examinations_pivoted_df['BP_Diastolic'], errors='coerce')

examinations_pivoted_df['BP_Systolic'] = examinations_pivoted_df.apply(lambda row: bp_systolic(row), axis=1)
examinations_pivoted_df['BP_Systolic'] = pd.to_numeric(examinations_pivoted_df['BP_Systolic'], errors='coerce')

examinations_pivoted_df.insert(4, 'Arterial_Pressure', examinations_pivoted_df.apply(lambda row: arterial_pressure(row), axis=1))

del bp_diasyloic, bp_systolic, arterial_pressure

### Structure & Populate Height, Weight and BMI Results

In [ ]:
def bmi_calc(row):
    if (pd.isnull(row['BMI']) or row['BMI']==0) and not pd.isnull(row['Height']) and not pd.isnull(row['Weight']):
        return float(row['Weight'])/((float(row['Height'])/100)**2)
    elif row['BMI'] == 0:
        return np.NaN
    else:
        return row['BMI']

In [ ]:
def find_weight(row):
    if not pd.isnull(row['BMI']) and not pd.isnull(row['Height']) and pd.isnull(row['Weight']):
        return float(row['BMI'])*((float(row['Height'])/100)**2)
    else:
        return row['Weight']

In [ ]:
def find_height(row):
    if not pd.isnull(row['BMI']) and pd.isnull(row['Height']) and not pd.isnull(row['Weight']):
        return sqrt(float(row['Weight']) / float(row['BMI'])) * 100
    else:
        return row['Height']

In [ ]:
examinations_pivoted_df['BMI'] = pd.to_numeric(examinations_pivoted_df['BMI'], errors='coerce')
examinations_pivoted_df['BMI'] = examinations_pivoted_df.apply(lambda row: bmi_calc(row), axis=1)

examinations_pivoted_df['Weight'] = pd.to_numeric(examinations_pivoted_df['Weight'], errors='coerce')
examinations_pivoted_df['Weight'] = examinations_pivoted_df.apply(lambda row: find_weight(row), axis=1)

examinations_pivoted_df['Height'] = pd.to_numeric(examinations_pivoted_df['Height'], errors='coerce')
examinations_pivoted_df['Height'] = examinations_pivoted_df.apply(lambda row: find_height(row), axis=1)

del bmi_calc, find_weight, find_height

In [ ]:
cols_to_drop = ['Blood_Pressure_Breakdown', 'Blood_Pressure']

examinations_pivoted_df = examinations_pivoted_df.drop(columns=cols_to_drop)

## Aggregate Results

In [ ]:
group_cols = list(examinations_pivoted_df.columns[:2])
agg = {}

for col in examinations_pivoted_df.columns[3:]:
     agg[col] = 'mean'

aggregated_examinations_df = examinations_pivoted_df.groupby(group_cols).aggregate(agg).reset_index()

del examinations_pivoted_df, group_cols, agg

In [ ]:
full_examinations_df = dates_breakdown_df.merge(aggregated_examinations_df, how='left', on=['master_person_id', 'grouping'])

del dates_breakdown_df, aggregated_examinations_df

full_examinations_df.head()

## Export Examinations Data

In [ ]:
# --- Save Results ---
file_name = "20260306_processed_examinations.csv"

full_examinations_df.to_csv(f"{processed_data_path}/{file_name}", index=False)
print("✅ Results saved.")